# Merging States

The parsers generated by `Larkita` are *canonical* `LR(1)` parsers.  Canonical `LR(1)` automata often have many
more states than `LALR(1)` automata, because they keep apart states that only differ in the look-ahead sets of
their marked rules.  Such states have the same *core*.  With the argument `~merge_states:true`, `Larkita`
merges states with the same core, as long as this is safe:

1. Merging must not cause a new conflict, neither in the merged states nor in their successors.
2. Merging must not change how the contextual scanner splits the input into tokens.

Then the resulting parser accepts the same strings and returns the same abstract syntax trees as the canonical
parser.

## A Grammar for Arithmetic Expressions

We use the grammar from the notebook [02-Conflicts-Resolved.ipynb](02-Conflicts-Resolved.ipynb).

In [ ]:
#use "../lib/nb.ml";;
load_lib "larkita.ml";;
cat "conflicts-resolved.lark"

We generate a canonical parser and a parser with merged states and compare the numbers of their states.

In [ ]:
let file_name = "conflicts-resolved.lark"
let canonical = Larkita.open_ ~out_file:"arith.out" file_name
let merged    = Larkita.open_ ~out_file:"arith-merged.out" ~merge_states:true file_name
;;
Printf.printf "canonical: %d states\n" (Array.length (Larkita.parse_table canonical).table);
Printf.printf "merged:    %d states\n%!" (Array.length (Larkita.parse_table merged).table)

Both parsers return the same abstract syntax trees.

In [ ]:
List.iter
  (fun s ->
    assert (Larkita.parse canonical s = Larkita.parse merged s);
    print_endline (Larkita.to_string (Larkita.parse merged s)))
  [ "1+2*3^4"; "2^3^4*5+6-7/8^9"; "(1+2)*3" ]

The output file of a parser with merged states contains the section `Merged states`.  It explains which states
with the same core have been kept apart.  The function `show_merged_states` prints this section.

In [ ]:
(* report_index report sub is the position of the first occurrence of sub in report, like report.index(sub) *)
let report_index (report : string) (sub : string) : int =
  let n = String.length sub in
  let rec find i = if String.sub report i n = sub then i else find (i + 1) in
  find 0

(* the part of report between the strings start and stop, like report[report.index(start):report.index(stop)] *)
let report_slice (report : string) (start : string) (stop : string) : string =
  let i = report_index report start in
  String.sub report i (report_index report stop - i)

(* split s sep splits the string s at the occurrences of the string sep, like s.split(sep) *)
let split (s : string) (sep : string) : string list =
  let n = String.length sep in
  let rec go start i acc =
    if i + n > String.length s then List.rev (String.sub s start (String.length s - start) :: acc)
    else if String.sub s i n = sep then go (i + n) (i + n) (String.sub s start (i - start) :: acc)
    else go start (i + 1) acc
  in
  go 0 0 []

(* contains s sub checks whether sub occurs in s, like sub in s *)
let contains (s : string) (sub : string) : bool =
  let n = String.length sub in
  let rec go i = i + n <= String.length s && (String.sub s i n = sub || go (i + 1)) in
  go 0

In [ ]:
let show_merged_states (out_file : string) : unit =
  let report = read_file out_file in
  print_endline (String.trim (report_slice report "Merged states" "\nStates\n======"))

In [ ]:
show_merged_states "arith-merged.out"

For this grammar, all states with the same core have been merged.  Hence, the merged automaton is the `LALR(1)`
automaton.  Every merged state lists the states of the canonical automaton that it has been merged from, and the
look-ahead set of every marked rule is the union of the look-ahead sets of this rule in these states.  The function
`show_state` prints the states that contain a given marked rule.

In [ ]:
let show_state (out_file : string) (marked_rule : string) : unit =
  let report = read_file out_file in
  List.iter
    (fun state -> if contains state marked_rule then print_endline ("State " ^ String.trim state ^ "\n"))
    (List.tl (split report "\n\n\nState "))

In [ ]:
show_state "arith-merged.out" "atom → NUMBER •"

In the canonical automaton, the marked rule `atom → NUMBER •` occurs in two states:  The look-ahead set of the first
state contains `$END`, while the look-ahead set of the second state contains `")"`, because the second state is
reached inside parentheses.

In [ ]:
show_state "arith.out" "atom → NUMBER •"

## The Grammar with the Mysterious Conflict

The grammar from the notebook [05-Mysterious-Conflicts.ipynb](05-Mysterious-Conflicts.ipynb) is an `LR(1)`
grammar, but it is not an `LALR(1)` grammar:  Merging the two states that contain the marked rule `a → X •`
would cause two reduce/reduce conflicts.  Therefore, `Larkita` keeps these states apart.

In [ ]:
cat "mysterious.lark"

In [ ]:
let parser = Larkita.open_ ~out_file:"mysterious-merged.out" ~merge_states:true "mysterious.lark"
;;
show_merged_states "mysterious-merged.out"

All four strings of the language are still parsed correctly.

In [ ]:
List.iter (fun s -> Printf.printf "%s %s\n%!" s (Larkita.to_string (Larkita.parse parser s))) [ "vxy"; "vxz"; "wxy"; "wxz" ]